# Review windows from a labeling run

Page through the windows produced by `scripts/build_dataset.py`, note the bad ones,
and write `exclude.txt` in the run directory. `scripts/merge_datasets.py` drops those
indexes automatically.

Budget about one afternoon per new class. For catalog types with weaker labels
(ice quake, landslide) this step is not optional.

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
sys.path.insert(0, str(Path.cwd().resolve().parents[1]))
from src.data.windows import load_dataset

# ── pick the run to review ───────────────────────────────────────────────
RUNS = sorted(Path("labeled_data/runs").glob("*"))
print("\n".join(f"[{i}] {r.name}" for i, r in enumerate(RUNS)))
RUN = RUNS[-1]          # <-- change index if needed
X, y, meta, info = load_dataset(RUN)
classes = info["classes"]
fs = info["window"]["sampling_rate"]
print(f"\n{RUN.name}: {len(X)} windows, classes {classes}")
print(meta["label_name"].value_counts())

In [ ]:
# ── browse ───────────────────────────────────────────────────────────────
LABEL   = None      # class name (e.g. "Blast") or None for all
WTYPE   = "event"   # "event", "noise" or None
PAGE    = 0
PER_ROW, ROWS = 4, 4

mask = np.ones(len(X), bool)
if LABEL is not None:
    mask &= (meta["label_name"] == LABEL).values
if WTYPE is not None:
    mask &= (meta["window_type"] == WTYPE).values
idx = np.where(mask)[0]
start = PAGE * PER_ROW * ROWS
batch = idx[start:start + PER_ROW * ROWS]

fig, axes = plt.subplots(ROWS, PER_ROW, figsize=(4 * PER_ROW, 2.2 * ROWS))
for ax, i in zip(axes.flat, batch):
    w = X[i]; t = np.arange(len(w)) / fs
    ax.plot(t, w, lw=0.5, color="k")
    m = meta.iloc[i]
    if m["window_type"] == "event":
        ax.axvline(m["onset_sample"] / fs, color="r", lw=0.8)
    ax.set_title(f"[{i}] {m['label_name']} {m['station']} M{m.get('magnitude', float('nan')):.1f} "
                 f"{m.get('distance_km', float('nan')):.0f}km {m.get('onset_source', '')}", fontsize=8)
    ax.tick_params(labelsize=6)
for ax in axes.flat[len(batch):]:
    ax.axis("off")
plt.suptitle(f"{RUN.name}  showing {start + 1}-{start + len(batch)} of {len(idx)}  (PAGE={PAGE})", fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# ── mark bad windows ─────────────────────────────────────────────────────
# Append indexes here as you page. Run this cell at the end to write exclude.txt.
BAD = [
    # 12, 47,
]
ex = RUN / "exclude.txt"
existing = {int(v) for v in ex.read_text().split()} if ex.exists() else set()
existing |= set(BAD)
ex.write_text("\n".join(str(i) for i in sorted(existing)) + ("\n" if existing else ""))
print(f"{len(existing)} excluded indexes written to {ex}")

In [ ]:
# ── quick class sanity: mean spectrum per class ──────────────────────────
from scipy import signal as sps
fig, ax = plt.subplots(figsize=(7, 4))
for c in classes:
    sel = np.where((meta["label_name"] == c).values)[0]
    if len(sel) == 0:
        continue
    psd = []
    for i in sel[:200]:
        f, p = sps.welch(X[i], fs=fs, nperseg=512)
        psd.append(p)
    ax.semilogy(f, np.median(psd, axis=0), label=f"{c} (n={len(sel)})")
ax.set_xlabel("Hz"); ax.set_ylabel("PSD"); ax.set_xlim(0, 30); ax.legend()
plt.title("Median spectrum per class (should differ if the classes are learnable)")
plt.show()